# AdaptCLIP Bottle CSV Failure Analysis

This notebook analyzes one comparison CSV: `results/adaptclip/bottle.csv`.

Flow:
- load sample scores
- defect-type AUROC table
- score distribution by defect type
- normal/good score variation with KDE
- high-score normal and low-score anomaly samples
- defect-type score variance
- normal variation difficulty


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
CSV_PATH = ROOT / 'results' / 'adaptclip' / 'bottle.csv'
MODEL_NAME = 'adaptclip'
TARGET_DATASET = 'mvtec'
TARGET_CLASS = 'bottle'

LOW_AUROC_THRESHOLD = 0.80
HIGH_AUROC_THRESHOLD = 0.90
TOPK_SAMPLES = 10

OUT_DIR = ROOT / 'results' / 'defect_type_analysis' / 'by_class' / f'{TARGET_DATASET}_{TARGET_CLASS}_{MODEL_NAME}_csv'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


## 1. Load Scores

In [ ]:
def normalize_path(path):
    return str(path).replace('\\', '/').replace('\\', '/')


def defect_type_from_path(path):
    parts = normalize_path(path).split('/')

    # MVTec: class/test/defect_type/image.png
    if 'test' in parts:
        idx = parts.index('test')
        if idx + 1 < len(parts):
            return parts[idx + 1]

    # ViSA: class/Data/Images/Normal or Anomaly/image.JPG
    if 'Images' in parts:
        idx = parts.index('Images')
        if idx + 1 < len(parts):
            defect_type = parts[idx + 1]
            return 'good' if defect_type.lower() == 'normal' else defect_type

    return 'unknown'


target = pd.read_csv(CSV_PATH)
target['model'] = MODEL_NAME
target['dataset'] = target['dataset'].astype(str).str.lower()
target['class'] = target['class'].astype(str)
target['label'] = target['label'].astype(int)
target['image_score'] = target['image_score'].astype(float)
target['sample_id'] = target['sample_id'].astype(str).str.zfill(3)
target['path_norm'] = target['query_path'].map(normalize_path)
target['defect_type'] = target['query_path'].map(defect_type_from_path)

if TARGET_DATASET is not None:
    target = target[target['dataset'] == TARGET_DATASET.lower()].copy()
if TARGET_CLASS is not None:
    target = target[target['class'] == TARGET_CLASS].copy()

if target.empty:
    raise ValueError(f'No rows found in {CSV_PATH} for {TARGET_DATASET}/{TARGET_CLASS}')

print('csv:', CSV_PATH)
print('rows:', len(target))
print('models:', sorted(target['model'].unique()))
print('defect types:', sorted(target['defect_type'].unique()))

target.groupby(['model', 'defect_type', 'label']).size().rename('n').reset_index()


## 2. Defect-Type AUROC and Summary

Each defect-type AUROC compares that defect type against `good` samples from the same class and model.

In [ ]:
rows = []

for model, model_df in target.groupby('model', sort=True):
    good = model_df[model_df['label'] == 0]
    anomalies = model_df[model_df['label'] == 1]

    class_auroc = roc_auc_score(model_df['label'], model_df['image_score']) if model_df['label'].nunique() == 2 else np.nan

    for defect_type, defect_df in anomalies.groupby('defect_type', sort=True):
        eval_df = pd.concat([good, defect_df], ignore_index=True)
        defect_auroc = roc_auc_score(eval_df['label'], eval_df['image_score']) if eval_df['label'].nunique() == 2 else np.nan

        rows.append({
            'model': model,
            'dataset': TARGET_DATASET.lower(),
            'class': TARGET_CLASS,
            'defect_type': defect_type,
            'class_auroc': class_auroc * 100,
            'defect_auroc': defect_auroc * 100,
            'n_good': len(good),
            'n_defect': len(defect_df),
            'mean_good_score': good['image_score'].mean(),
            'mean_defect_score': defect_df['image_score'].mean(),
            'score_gap': defect_df['image_score'].mean() - good['image_score'].mean(),
            'std_good_score': good['image_score'].std(ddof=1),
            'std_defect_score': defect_df['image_score'].std(ddof=1),
            'iqr_defect_score': defect_df['image_score'].quantile(0.75) - defect_df['image_score'].quantile(0.25),
            'range_defect_score': defect_df['image_score'].max() - defect_df['image_score'].min(),
            'defect_to_good_std_ratio': defect_df['image_score'].std(ddof=1) / good['image_score'].std(ddof=1),
            'top_normal_score': good['image_score'].max(),
            'bottom_anomaly_score': defect_df['image_score'].min(),
        })

summary = pd.DataFrame(rows)
summary['rank_low_to_high'] = summary.groupby('model')['defect_auroc'].rank(ascending=True, method='min').astype(int)
summary['low_auroc'] = summary['defect_auroc'].le(LOW_AUROC_THRESHOLD * 100)
summary = summary.sort_values(['model', 'defect_auroc', 'defect_type']).reset_index(drop=True)
summary.to_csv(OUT_DIR / 'class_defect_type_summary.csv', index=False)

summary_display = summary.copy()
round_cols = [col for col in summary_display.columns if col.endswith('_score') or col in ['class_auroc', 'defect_auroc', 'score_gap']]
summary_display[round_cols] = summary_display[round_cols].round(4)
summary_display


## 3. Defect-Type AUROC Bar Plot

In [ ]:
plot_df = summary_display.sort_values(['model', 'defect_auroc']).copy()

fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(plot_df))))
if plot_df.empty:
    ax.axis('off')
    ax.set_title('No defect AUROC data')
else:
    colors = np.where(plot_df['low_auroc'], '#d55e00', '#0072b2')
    ax.bar(plot_df['defect_type'], plot_df['defect_auroc'], color=colors)
    ax.axhline(LOW_AUROC_THRESHOLD * 100, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{MODEL_NAME}: {TARGET_DATASET}/{TARGET_CLASS}')
    ax.set_ylabel('AUROC (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha('right')
plt.tight_layout()
plt.show()


## 4. Score Distribution by Defect Type

The boxplot shows normal/anomaly score overlap by defect type. The normal-only density plot focuses on how widely good scores spread.

In [ ]:
def kde_curve(values, xlim=None, n_grid=256):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 2 or np.nanstd(values) == 0:
        return None, None

    std = values.std(ddof=1)
    bandwidth = 1.06 * std * values.size ** (-1 / 5)
    if bandwidth <= 0 or not np.isfinite(bandwidth):
        return None, None

    if xlim is None:
        pad = 3 * bandwidth
        xs = np.linspace(values.min() - pad, values.max() + pad, n_grid)
    else:
        xs = np.linspace(xlim[0], xlim[1], n_grid)
    density = np.exp(-0.5 * ((xs[:, None] - values[None, :]) / bandwidth) ** 2).sum(axis=1)
    density /= values.size * bandwidth * np.sqrt(2 * np.pi)
    return xs, density


all_scores = target['image_score'].dropna().to_numpy()
score_xmin = np.nanmin(all_scores)
score_xmax = np.nanmax(all_scores)
score_xpad = max((score_xmax - score_xmin) * 0.08, 1e-6)
score_xlim = (score_xmin - score_xpad, score_xmax + score_xpad)
score_bins = np.linspace(score_xlim[0], score_xlim[1], 31)

model_df = target[target['model'] == MODEL_NAME].copy()
order = ['good'] + sorted([x for x in model_df['defect_type'].unique() if x != 'good'])
data = [model_df[model_df['defect_type'] == defect]['image_score'].values for defect in order]

fig, ax = plt.subplots(figsize=(max(7, len(order) * 0.8), 3.6))
ax.boxplot(data, tick_labels=order, showfliers=True)
ax.set_title(f'{MODEL_NAME}: score distribution by defect type')
ax.set_ylabel('image_score')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.show()

normal_scores = model_df[model_df['label'] == 0]['image_score'].dropna().to_numpy()
if normal_scores.size:
    mean_score = normal_scores.mean()
    median_score = np.median(normal_scores)
    p10, p90 = np.quantile(normal_scores, [0.10, 0.90])
    p25, p75 = np.quantile(normal_scores, [0.25, 0.75])

    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.hist(normal_scores, bins=score_bins, density=True, alpha=0.35, color='#4c78a8', edgecolor='white')
    xs, density = kde_curve(normal_scores, xlim=score_xlim)
    if xs is not None:
        ax.plot(xs, density, color='#1f4e79', linewidth=2, label='KDE')

    ax.axvspan(p25, p75, color='#59a14f', alpha=0.16, label='IQR')
    ax.axvline(mean_score, color='#e15759', linestyle='--', linewidth=1.5, label='mean')
    ax.axvline(median_score, color='#111111', linestyle=':', linewidth=1.5, label='median')
    ax.axvline(p90, color='#f28e2b', linestyle='-.', linewidth=1.5, label='p90')
    ax.set_title(f'{MODEL_NAME}: normal/good score variation')
    ax.set_xlabel('image_score')
    ax.set_ylabel('density')
    ax.set_xlim(score_xlim)
    ax.legend(frameon=False, ncol=4, fontsize=8)
    ax.text(
        0.01, 0.98,
        f'n={normal_scores.size} | std={normal_scores.std(ddof=1):.4f} | IQR={(p75 - p25):.4f} | p90-p10={(p90 - p10):.4f}',
        transform=ax.transAxes, va='top', ha='left', fontsize=9,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75),
    )
    plt.tight_layout()
    plt.show()


## 5. Hard Normal and Missed Anomaly Samples

- High-score normal: likely false-positive normal variation.
- Low-score anomaly: likely false-negative or weakly activated anomaly.

In [ ]:
def top_hard_normal(n=TOPK_SAMPLES):
    cols = ['model', 'dataset', 'class', 'defect_type', 'sample_id', 'image_score', 'query_path']
    return (
        target[target['label'] == 0]
        .sort_values('image_score', ascending=False)[cols]
        .head(n)
        .reset_index(drop=True)
    )


def top_missed_anomaly(n=TOPK_SAMPLES):
    cols = ['model', 'dataset', 'class', 'defect_type', 'sample_id', 'image_score', 'query_path']
    return (
        target[target['label'] == 1]
        .sort_values('image_score', ascending=True)[cols]
        .head(n)
        .reset_index(drop=True)
    )


hard_normal = top_hard_normal()
missed_anomaly = top_missed_anomaly()

hard_normal.to_csv(OUT_DIR / f'{MODEL_NAME}_hard_normal_top{TOPK_SAMPLES}.csv', index=False)
missed_anomaly.to_csv(OUT_DIR / f'{MODEL_NAME}_missed_anomaly_top{TOPK_SAMPLES}.csv', index=False)
print('Saved hard normal and missed anomaly tables to:', OUT_DIR)


In [ ]:
def show_images(df, title, n=TOPK_SAMPLES):
    show_df = df.head(n).copy()
    if show_df.empty:
        print('No samples:', title)
        return

    ncols = min(5, len(show_df))
    nrows = int(np.ceil(len(show_df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis('off')

    for ax, (_, row) in zip(axes, show_df.iterrows()):
        image_path = ROOT / normalize_path(row['query_path'])
        if image_path.exists():
            image = plt.imread(image_path)
            ax.imshow(image, cmap='gray' if image.ndim == 2 else None)
        else:
            ax.text(0.5, 0.5, 'missing image', ha='center', va='center')
        ax.set_title(f"{row['defect_type']} | {row['sample_id']}\nscore={row['image_score']:.4f}", fontsize=9)

    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


print('high-score normal samples')
display(hard_normal)
show_images(hard_normal, f'{MODEL_NAME}: high-score normal samples')

print('low-score anomaly samples')
display(missed_anomaly)
show_images(missed_anomaly, f'{MODEL_NAME}: low-score anomaly samples')


## 6. Defect-Type Variance and Unstable Defects

Large within-defect variance means the model detects some samples of the same defect type but misses others.

In [ ]:
variance_cols = [
    'model', 'defect_type', 'defect_auroc', 'n_defect', 'mean_defect_score',
    'std_defect_score', 'iqr_defect_score', 'range_defect_score', 'bottom_anomaly_score'
]
variance_table = summary_display[variance_cols].copy()
variance_table = variance_table.sort_values(['model', 'std_defect_score'], ascending=[True, False]).reset_index(drop=True)
variance_table.to_csv(OUT_DIR / 'defect_type_variance.csv', index=False)
variance_table


In [ ]:
model_df = variance_table.sort_values('std_defect_score')
fig, ax = plt.subplots(figsize=(8, max(3, len(model_df) * 0.35)))
colors = np.where(model_df['defect_auroc'] <= LOW_AUROC_THRESHOLD * 100, '#d55e00', '#0072b2')
ax.barh(model_df['defect_type'], model_df['std_defect_score'], color=colors)
ax.set_title(f'{MODEL_NAME}: within-defect score std')
ax.set_xlabel('std of anomaly scores')
plt.tight_layout()
plt.show()


## 7. Normal Variation Difficulty

High normal scores indicate normal variation that looks suspicious to the model. Wider std, IQR, MAD, or p90-p10 means the normal score distribution is more spread out.

In [ ]:
def median_abs_deviation(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    median = np.median(values)
    return np.median(np.abs(values - median))


normal_difficulty = (
    target[target['label'] == 0]
    .groupby('model')
    .agg(
        n_normal=('image_score', 'size'),
        mean_normal_score=('image_score', 'mean'),
        median_normal_score=('image_score', 'median'),
        std_normal_score=('image_score', 'std'),
        mad_normal_score=('image_score', median_abs_deviation),
        min_normal_score=('image_score', 'min'),
        p10_normal_score=('image_score', lambda x: x.quantile(0.10)),
        p25_normal_score=('image_score', lambda x: x.quantile(0.25)),
        p75_normal_score=('image_score', lambda x: x.quantile(0.75)),
        p90_normal_score=('image_score', lambda x: x.quantile(0.90)),
        p95_normal_score=('image_score', lambda x: x.quantile(0.95)),
        max_normal_score=('image_score', 'max'),
    )
    .reset_index()
)
normal_difficulty['iqr_normal_score'] = normal_difficulty['p75_normal_score'] - normal_difficulty['p25_normal_score']
normal_difficulty['p90_p10_normal_score'] = normal_difficulty['p90_normal_score'] - normal_difficulty['p10_normal_score']
normal_difficulty['max_median_normal_score'] = normal_difficulty['max_normal_score'] - normal_difficulty['median_normal_score']

display_cols = [
    'model', 'n_normal', 'mean_normal_score', 'median_normal_score',
    'std_normal_score', 'mad_normal_score', 'iqr_normal_score',
    'p90_p10_normal_score', 'p90_normal_score', 'p95_normal_score',
    'max_normal_score', 'max_median_normal_score',
]
round_cols = [col for col in normal_difficulty.columns if col.endswith('_score')]
normal_difficulty[round_cols] = normal_difficulty[round_cols].round(4)
normal_difficulty.to_csv(OUT_DIR / 'normal_variation_difficulty.csv', index=False)
normal_difficulty[display_cols]


## 8. Output Files

Generated CSVs are saved under `results/defect_type_analysis/by_class/mvtec_bottle_adaptclip_csv/`:

- `class_defect_type_summary.csv`
- `adaptclip_hard_normal_top10.csv`
- `adaptclip_missed_anomaly_top10.csv`
- `defect_type_variance.csv`
- `normal_variation_difficulty.csv`
